<a href="https://colab.research.google.com/github/mdzikrim/MachineLearningClass/blob/main/Chapter_12_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Machine Learning Chapter 12

1. How would you describe TensorFlow in a short sentence? What are its main features? Can you name other popular Deep Learning libraries?
> TensorFlow adalah library open-source untuk komputasi numerik yang sangat cocok untuk ML skala besar. Mampu mendukung GPU, komputasi terdistribusi, analisis/optimasi computation graph (format portable), reverse-mode autodiff untuk optimasi, serta API seperti tf.keras, tf.data, tf.image, tf.signal, dll

2. Is TensorFlow a drop-in replacement for NumPy? What are the main differences between the two?
> Tidak sepenuhnya. Nama fungsi sering berbeda (mis. tf.reduce_sum() vs np.sum()), sebagian fungsi berperilaku berbeda (mis. tf.transpose() membuat salinan, sedangkan ndarray.T umumnya view), dan NumPy array mutable sementara tensor TF tidak (pakai tf.Variable bila butuh mutable).

3. Do you get the same result with tf.range(10) and tf.constant(np.arange(10))?
> Nilainya sama (0–9), tetapi tipe default berbeda: tf.range(10) default int32, sedangkan NumPy default int64 sehingga tf.constant(np.arange(10)) menjadi int64.

4. Can you name six other data structures available in TensorFlow, beyond regular tensors?
> Sparse tensors, tensor arrays, ragged tensors, queues, string tensors, sets.

5. A custom loss function can be defined by writing a function or by subclassing the keras.losses.Loss class. When would you use each option?
> Biasanya cukup fungsi Python. Jika loss perlu hyperparameter/state, subclass keras.losses.Loss dan implement __init__() + call(). Jika ingin hyperparameter tersimpan bersama model, implement juga get_config().

6. Similarly, a custom metric can be defined in a function or a subclass of keras.metrics.Metric. When would you use each option?
> Biasanya cukup fungsi. Jika butuh hyperparameter/state, atau jika metrik “epoch-wise” tidak sama dengan rata-rata metrik per batch (mis. precision/recall), maka subclass keras.metrics.Metric dan implement __init__(), update_state(), result() (dan reset_states() bila perlu), serta get_config() jika ingin state tersimpan.

7. When should you create a custom layer versus a custom model?
> Pisahkan komponen internal (layer / blok reusable) dari model (objek yang ditraining). Komponen internal subclass keras.layers.Layer, sedangkan model subclass keras.models.Model.

8. What are some use cases that require writing your own custom training loop?
> Custom training loop itu advanced; gunakan bila benar-benar perlu. Keras menyediakan banyak cara kustomisasi tanpa custom loop (callbacks, regularizers, constraints, custom losses, dll). Namun custom loop diperlukan misalnya jika ingin optimizer berbeda untuk bagian berbeda dari network (contoh Wide & Deep), atau untuk debugging/memahami detail training.

9. Can custom Keras components contain arbitrary Python code, or must they be convertible to TF Functions?
> Idealnya harus convertible ke TF Functions (utamakan operasi TF dan ikuti aturan TF Function). Jika butuh Python arbitrary, bisa bungkus dengan tf.py_function() (mengurangi performa & portabilitas) atau buat model/layer dynamic (dynamic=True) / compile(run_eagerly=True).

10. What are the main rules to respect if you want a function to be convertible to a TF Function?
> - Panggilan ke library non-TF (NumPy/stdlib) hanya berjalan saat tracing, bukan bagian graph; side effects juga hanya saat tracing; tf.py_function() memungkinkan Python arbitrary tetapi menghambat optimasi graph.
> - Pembuatan tf.Variable/objek stateful harus hanya pada call pertama (lebih baik dibuat di luar TF Function, mis. build() layer). Update variabel pakai assign(), bukan =.
> - Source code fungsi harus tersedia untuk AutoGraph/tracing.
> - Loop yang ingin “ditangkap” graph harus iterasi atas tensor/dataset (mis. for i in tf.range(x) bukan range(x)), dan prefer vectorization untuk performa.

11. When would you need to create a dynamic Keras model? How do you do that? Why not make all your models dynamic?
> Dynamic berguna untuk debugging (karena tidak dikompilasi ke TF Function; bisa pakai debugger Python) atau jika perlu arbitrary Python/external libs di model/training. Cara: dynamic=True saat membuat custom layer/model, atau compile(run_eagerly=True). Tidak semua dibuat dynamic karena akan memperlambat training/inference dan menghilangkan manfaat graph + mengurangi portabilitas (sulit diexport graph).

12. Implement a custom layer that performs Layer Normalization (we will use this type of layer in Chapter 15):

In [2]:
import tensorflow as tf
from tensorflow import keras

In [4]:
class MyLayerNormalization(keras.layers.Layer):
    def __init__(self, epsilon=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        param_shape = input_shape[-1:]
        self.alpha = self.add_weight(
            name="alpha",
            shape=param_shape,
            dtype=tf.float32,
            initializer="ones",
            trainable=True,
        )
        self.beta = self.add_weight(
            name="beta",
            shape=param_shape,
            dtype=tf.float32,
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        mean, variance = tf.nn.moments(inputs, axes=-1, keepdims=True)
        std = tf.sqrt(variance)
        normalized = (inputs - mean) / (std + self.epsilon)
        return self.alpha * normalized + self.beta

    def get_config(self):
        config = super().get_config()
        config.update({"epsilon": self.epsilon})
        return config

In [5]:
X = tf.random.normal(shape=(4, 10), dtype=tf.float32)

ref = keras.layers.LayerNormalization(epsilon=1e-3)
mine = MyLayerNormalization(epsilon=1e-3)

Y_ref = ref(X)
Y_mine = mine(X)

max_abs_diff = tf.reduce_max(tf.abs(Y_ref - Y_mine))
print("max_abs_diff:", float(max_abs_diff.numpy()))

max_abs_diff: 0.0012347698211669922


13. Train a model using a custom training loop to tackle the Fashion MNIST dataset (see Chapter 10).

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
np.random.seed(42)

In [7]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

X_valid, y_valid = X_train_full[:5000], y_train_full[:5000]
X_train, y_train = X_train_full[5000:], y_train_full[5000:]

batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(20000).batch(batch_size).prefetch(1)
valid_ds = tf.data.Dataset.from_tensor_slices((X_valid, y_valid)).batch(batch_size).prefetch(1)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
lower = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
], name="lower")

upper = keras.Sequential([
    keras.layers.Dense(10, activation="softmax"),
], name="upper")

class MyModel(keras.Model):
    def __init__(self, lower, upper):
        super().__init__()
        self.lower = lower
        self.upper = upper

    def call(self, inputs, training=False):
        x = self.lower(inputs, training=training)
        return self.upper(x, training=training)

model = MyModel(lower, upper)

In [9]:
loss_fn = keras.losses.SparseCategoricalCrossentropy()
train_loss = keras.metrics.Mean(name="train_loss")
train_acc = keras.metrics.SparseCategoricalAccuracy(name="train_accuracy")
valid_loss = keras.metrics.Mean(name="valid_loss")
valid_acc = keras.metrics.SparseCategoricalAccuracy(name="valid_accuracy")

In [26]:
opt_lower = keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9)
opt_upper = keras.optimizers.SGD(learning_rate=5e-2, momentum=0.9)

def print_status_bar(iteration, total, metrics):
    metrics_str = " - ".join([f"{m.name}: {m.result():.4f}" for m in metrics])
    end = "" if iteration < total else "\n"
    print(f"\r{iteration}/{total} - {metrics_str}", end=end)

@tf.function
def train_step(Xb, yb):
    with tf.GradientTape() as tape:
        y_pred = model(Xb, training=True)
        loss = loss_fn(yb, y_pred)
    # grads for all trainable vars in the order they appear in model.trainable_variables
    all_grads = tape.gradient(loss, model.trainable_variables)

    lower_vars = model.lower.trainable_variables
    upper_vars = model.upper.trainable_variables

    # Split `all_grads` based on the number of variables in each sub-model.
    # This assumes that model.trainable_variables is the concatenation of
    # lower_vars and upper_vars in that order.
    num_lower_vars = len(lower_vars)
    lower_grads = all_grads[:num_lower_vars]
    upper_grads = all_grads[num_lower_vars:]

    opt_lower.apply_gradients(zip(lower_grads, lower_vars))
    opt_upper.apply_gradients(zip(upper_grads, upper_vars))

    train_loss.update_state(loss)
    train_acc.update_state(yb, y_pred)

In [27]:
@tf.function
def valid_step(Xb, yb):
    y_pred = model(Xb, training=False)
    vloss = loss_fn(yb, y_pred)
    valid_loss.update_state(vloss)
    valid_acc.update_state(yb, y_pred)

# Training loop (a)
epochs = 10
for epoch in range(1, epochs + 1):
    print(f"Epoch {epoch}/{epochs}")

    # reset metrics
    # train_loss.reset_states() # Removed: tf.keras.metrics.Mean does not have reset_states()
    # train_acc.reset_states() # Removed: AttributeError in this environment
    # valid_loss.reset_states() # Removed: tf.keras.metrics.Mean does not have reset_states()
    # valid_acc.reset_states() # Removed: AttributeError in this environment

    # train
    n_steps = 0
    n_total = int(np.ceil(len(X_train) / batch_size))
    for step, (Xb, yb) in enumerate(train_ds, start=1):
        train_step(Xb, yb)
        n_steps = step
        print_status_bar(step, n_total, [train_loss, train_acc])

    # validation (end of epoch)
    for Xb, yb in valid_ds:
        valid_step(Xb, yb)

    print(f"val_loss: {valid_loss.result():.4f} - val_accuracy: {valid_acc.result():.4f}")

# Final test evaluation (optional quick check)
y_pred_test = model(tf.constant(X_test), training=False)
test_acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(y_pred_test, axis=1), y_test), tf.float32))
print("test_accuracy:", float(test_acc.numpy()))

Epoch 1/10
860/860 - train_loss: 0.5448 - train_accuracy: 0.8052
val_loss: 0.4109 - val_accuracy: 0.8468
Epoch 2/10
860/860 - train_loss: 0.4690 - train_accuracy: 0.8306
val_loss: 0.3832 - val_accuracy: 0.8596
Epoch 3/10
860/860 - train_loss: 0.4297 - train_accuracy: 0.8437
val_loss: 0.3693 - val_accuracy: 0.8648
Epoch 4/10
860/860 - train_loss: 0.4046 - train_accuracy: 0.8522
val_loss: 0.3556 - val_accuracy: 0.8698
Epoch 5/10
860/860 - train_loss: 0.3847 - train_accuracy: 0.8591
val_loss: 0.3488 - val_accuracy: 0.8724
Epoch 6/10
860/860 - train_loss: 0.3690 - train_accuracy: 0.8644
val_loss: 0.3437 - val_accuracy: 0.8741
Epoch 7/10
860/860 - train_loss: 0.3561 - train_accuracy: 0.8688
val_loss: 0.3369 - val_accuracy: 0.8764
Epoch 8/10
860/860 - train_loss: 0.3449 - train_accuracy: 0.8726
val_loss: 0.3305 - val_accuracy: 0.8784
Epoch 9/10
860/860 - train_loss: 0.3349 - train_accuracy: 0.8762
val_loss: 0.3264 - val_accuracy: 0.8797
Epoch 10/10
860/860 - train_loss: 0.3263 - train_accura